# Azure Monitor + Application Insights

These two services are closely related but have different responsibilities.

- **Azure Monitor** → the broader Azure monitoring and observability platform.
- **Application Insights** → application-performance monitoring (APM) capability within Azure Monitor.

---

# 1. Azure Monitor

**Azure Monitor** collects and analyzes telemetry from Azure resources, applications, infrastructure, and services.

It helps answer:

> **Is my system healthy? What is happening? Why is it happening?**

Typical telemetry:

```text
Metrics
Logs
Traces
Alerts
```

Architecture:

```text
Azure Resources
      │
      ├── Azure Functions
      ├── Azure OpenAI
      ├── AI Search
      ├── App Service
      └── Storage
             │
             ▼
        Azure Monitor
             │
      ┌──────┼──────┐
      ▼      ▼      ▼
   Metrics  Logs   Alerts
```

---

# 2. Application Insights

**Application Insights** is an Azure Monitor feature for monitoring applications.

It is particularly useful for understanding:

- Application requests
- Response times
- Exceptions
- Dependencies
- Distributed traces
- Availability
- Application performance

For a Python AI application:

```text
FastAPI / Azure Function
        │
        ▼
Application Insights
        │
        ▼
Azure Monitor
```

---

# 3. Azure Monitor vs Application Insights

| Azure Monitor | Application Insights |
|---|---|
| Broad Azure observability platform | Application-focused monitoring |
| Infrastructure + resources + applications | Application performance |
| Metrics | Requests |
| Logs | Exceptions |
| Alerts | Dependencies |
| Log Analytics | Distributed tracing |
| Resource health | Application telemetry |
| Dashboards | Application dashboards |

### Simple interview answer

> **"Azure Monitor is the overall observability platform, while Application Insights provides application performance monitoring capabilities within Azure Monitor."**

---

# 4. Why This Matters for AI Applications

Traditional application monitoring:

```text
Request
 ↓
API
 ↓
Database
 ↓
Response
```

AI application monitoring is more complex:

```text
User
 ↓
API
 ↓
Agent
 ↓
LLM
 ↓
Tool
 ↓
RAG
 ↓
Azure AI Search
 ↓
LLM
 ↓
Response
```

You need to understand **where latency, errors, and quality problems are occurring**.

---

# 5. Production AI Observability

For your Agentic AI architecture:

```text
                         User
                           │
                           ▼
                         APIM
                           │
                           ▼
                    AI Application
                           │
                           ▼
                      LangGraph
                           │
             ┌─────────────┼─────────────┐
             ▼             ▼             ▼
        Azure OpenAI   AI Search      Tool/API
             │             │             │
             └─────────────┼─────────────┘
                           ▼
                        Response
                           │
                           ▼
                  Application Insights
                           │
                           ▼
                     Azure Monitor
```

You can investigate the complete execution path.

---

# 6. What Should We Monitor?

For a production GenAI application, monitor several layers.

### Application

```text
Request count
Response time
HTTP errors
Exceptions
Availability
```

### LLM

```text
LLM latency
Model errors
Token usage
Input tokens
Output tokens
Model deployment
```

### RAG

```text
Retrieval latency
Number of retrieved chunks
Search errors
Top-K
Search scores
```

### Agent

```text
Agent execution time
Number of iterations
Tool calls
Tool failures
Workflow failures
```

### Infrastructure

```text
CPU
Memory
Network
Function executions
Storage
Service health
```

---

# 7. Example AI Request Trace

Suppose the user asks:

> "What is the company's leave policy?"

The request might travel through:

```text
Request ID: abc123

APIM
  ↓ 20 ms
FastAPI
  ↓ 10 ms
LangGraph
  ↓ 5 ms
Azure AI Search
  ↓ 180 ms
Azure OpenAI
  ↓ 900 ms
Content Safety
  ↓ 50 ms
Response
```

Total:

```text
~1165 ms
```

Application Insights/distributed tracing helps identify where the latency occurred.

---

# 8. Distributed Tracing ⭐⭐⭐⭐⭐

This is especially important in microservices and Agentic AI.

Instead of seeing:

```text
Request = 1.2 seconds
```

you want:

```text
Request
│
├── APIM             20 ms
│
├── Agent            15 ms
│
├── AI Search       180 ms
│
├── Azure OpenAI    900 ms
│
└── Safety           50 ms
```

This makes troubleshooting significantly easier.

---

# 9. Dependencies

Application Insights can track calls made by your application to dependencies.

Example:

```text
FastAPI
 │
 ├── Azure AI Search
 ├── Azure OpenAI
 ├── SQL
 ├── Redis
 └── External API
```

If Azure AI Search starts taking 5 seconds:

```text
Application
     ↓
Dependency telemetry
     ↓
AI Search latency ↑
```

you can identify the dependency causing the problem.

---

# 10. Logs

You can write application logs.

Python:

```python
import logging

logger = logging.getLogger(__name__)

logger.info("Starting RAG pipeline")
logger.warning("No documents retrieved")
logger.error("Azure AI Search failed")
```

These logs can be collected and queried through Azure monitoring capabilities.

---

# 11. Don't Log Sensitive Data

This is particularly important for AI systems.

Avoid:

```python
logger.info(f"User prompt: {prompt}")
logger.info(f"API key: {api_key}")
logger.info(f"Retrieved document: {full_document}")
```

Because prompts and retrieved documents may contain:

- PII
- Financial information
- Healthcare information
- Confidential enterprise information
- Credentials

Instead, log safe metadata:

```python
logger.info(
    "RAG request completed",
    extra={
        "request_id": request_id,
        "retrieval_count": len(docs),
        "latency_ms": latency
    }
)
```

---

# 12. Metrics

Metrics are numerical measurements over time.

Example:

```text
AI Request Count
        1000
        1200
        1500
```

Useful AI metrics:

| Metric | Why |
|---|---|
| Request count | Traffic |
| Error rate | Reliability |
| Latency | Performance |
| Token usage | Cost |
| Search latency | RAG performance |
| Tool failures | Agent reliability |
| Agent iterations | Workflow efficiency |
| HTTP 429 | Throttling |
| HTTP 5xx | Backend failure |

---

# 13. Alerts ⭐⭐⭐⭐⭐

Azure Monitor can trigger alerts when conditions are met.

Example:

```text
Error rate > 5%
        ↓
Azure Monitor Alert
        ↓
DevOps / Engineering Team
```

Other examples:

```text
LLM failures > threshold
Search latency > threshold
Function failures > threshold
CPU > threshold
Availability < threshold
```

---

# 14. AI-Specific Alerting

For Agentic AI, don't only monitor infrastructure.

Example:

```text
Agent Tool Failure Rate > 10%
             ↓
           Alert
```

or:

```text
Average LLM latency > 3 seconds
             ↓
           Alert
```

or:

```text
Azure OpenAI 429 responses increasing
             ↓
           Alert
```

This is more useful than simply monitoring CPU.

---

# 15. Application Insights + Azure Functions

This is a very common combination.

```text
Azure Function
      │
      ▼
Application Insights
      │
      ▼
Azure Monitor
```

Suppose your Function processes documents:

```text
Blob Upload
     ↓
Function
     ↓
Document Intelligence
     ↓
Embeddings
     ↓
AI Search
```

You can monitor:

```text
Function execution count
Execution duration
Failures
Dependencies
Exceptions
```

---

# 16. Application Insights + FastAPI

Your Python application can also emit telemetry.

Typical architecture:

```text
FastAPI
  │
  ├── LangChain
  ├── LangGraph
  ├── Azure OpenAI
  └── Azure AI Search
          │
          ▼
 Application Insights
          │
          ▼
    Azure Monitor
```

For Python applications, Azure Monitor's OpenTelemetry-based approach can be used to instrument applications and collect telemetry.

---

# 17. OpenTelemetry ⭐⭐⭐⭐⭐

**OpenTelemetry (OTel)** is an open standard/framework for collecting:

- Traces
- Metrics
- Logs

Conceptually:

```text
Application
    │
    ▼
OpenTelemetry
    │
    ▼
Azure Monitor / Application Insights
```

This is particularly useful in distributed AI systems.

---

# 18. LangGraph Observability

Suppose your graph is:

```text
START
 ↓
Classifier
 ↓
Retriever
 ↓
Tool
 ↓
Generator
 ↓
END
```

You want to know:

```text
Which node was slow?
Which tool failed?
How many times did the graph loop?
Why did the workflow terminate?
```

You can combine application telemetry with AI-specific observability tooling.

For example:

```text
LangGraph
   │
   ├── Node execution
   ├── Tool execution
   ├── Errors
   └── Latency
          │
          ▼
   Observability Layer
          │
          ▼
 Azure Monitor / App Insights
```

---

# 19. AI Cost Monitoring

This is a major production concern.

Suppose:

```text
1 request
↓
Agent
↓
3 LLM calls
↓
2 tool calls
↓
1 final LLM call
```

One user request may generate multiple model calls.

Track:

```text
Request
 ├── LLM Call 1 → 1,000 tokens
 ├── LLM Call 2 → 2,000 tokens
 └── LLM Call 3 → 1,500 tokens
```

Total:

```text
4,500 tokens
```

Then correlate:

```text
Request ID
+
Model
+
Token Usage
+
Latency
```

to understand cost and performance.

---

# 20. RAG Observability

For RAG:

```text
Question
 ↓
Embedding
 ↓
Azure AI Search
 ↓
Top-K
 ↓
Reranking
 ↓
Context
 ↓
LLM
```

Log safe metadata such as:

```python
{
    "request_id": "abc123",
    "top_k": 5,
    "retrieval_latency_ms": 180,
    "documents_retrieved": 5,
    "generation_latency_ms": 900
}
```

Avoid logging confidential document contents unless explicitly permitted.

---

# 21. Agent Observability

For an agent:

```text
Request ID
    │
    ▼
Agent
    │
    ├── LLM Call
    │
    ├── Search Tool
    │
    ├── API Tool
    │
    ├── LLM Call
    │
    └── Final Response
```

Useful metrics:

```text
Agent completion rate
Tool selection success
Tool failure rate
Average iterations
Average latency
LLM calls/request
Task success rate
```

---

# 22. Azure Monitor + APIM

We've already covered API Management.

Together:

```text
Client
  ↓
APIM
  ↓
FastAPI / Function
  ↓
Agent
  ↓
Azure OpenAI
```

Monitoring:

```text
APIM
 ↓
API metrics
 ↓
Azure Monitor

Application
 ↓
Application Insights
 ↓
Azure Monitor
```

This gives you two useful perspectives:

```text
APIM → API-level monitoring

App Insights → Application-level monitoring
```

---

# 23. Azure Monitor + Content Safety

You can also monitor safety-related events.

```text
User Request
     ↓
Content Safety
     ↓
Blocked?
     │
     ├── Yes → Log safety event
     │
     └── No → Continue
```

Track metrics such as:

```text
Safety blocks
Safety category
Blocked requests
Rate of blocked requests
```

Be careful about storing the actual sensitive content in logs.

---

# 24. Azure Monitor + Alerting Architecture

```text
                  Application
                      │
         ┌────────────┼─────────────┐
         ▼            ▼             ▼
       Logs        Metrics        Traces
         │            │             │
         └────────────┼─────────────┘
                      ▼
                Azure Monitor
                      │
              ┌───────┴────────┐
              ▼                ▼
          Dashboard          Alerts
                                │
                                ▼
                       Operations Team
```

---

# 25. Production AI Architecture

Putting everything together:

```text
                         USER
                           │
                           ▼
                    Microsoft Entra ID
                           │
                           ▼
                 Azure API Management
                           │
                           ▼
                    AI Application
                           │
                     LangGraph
                           │
          ┌────────────────┼─────────────────┐
          ▼                ▼                 ▼
     Azure OpenAI     Azure AI Search    Azure Functions
          │                │                 │
          └────────────────┼─────────────────┘
                           ▼
                    Content Safety
                           │
                           ▼
                       Response

────────────────────────────────────────────────

                 OBSERVABILITY
                       │
          ┌────────────┴────────────┐
          ▼                         ▼
 Application Insights          Azure Monitor
          │                         │
          ├── Requests              ├── Metrics
          ├── Dependencies          ├── Logs
          ├── Exceptions            ├── Alerts
          ├── Traces                └── Dashboards
          └── Performance
```

---

# 26. Important Interview Distinction

### Azure Monitor

> **"Azure Monitor is the broader observability platform for collecting and analyzing metrics, logs, traces, and alerts across Azure resources and applications."**

### Application Insights

> **"Application Insights is the application performance monitoring capability within Azure Monitor, providing application telemetry such as requests, dependencies, exceptions, availability, and distributed traces."**

---

# 27. Interview Questions

### Q1. What is Azure Monitor?

> Azure Monitor is Azure's observability platform for monitoring applications, infrastructure, and Azure resources using metrics, logs, traces, alerts, and dashboards.

### Q2. What is Application Insights?

> Application Insights is an Azure Monitor capability focused on application performance monitoring and telemetry.

### Q3. Difference between Azure Monitor and Application Insights?

> Azure Monitor is the broader platform, while Application Insights focuses primarily on application-level telemetry and performance.

### Q4. How would you monitor an Agentic AI application?

> "I would instrument the API, agent workflow, LLM calls, RAG retrieval, and tool calls. I would track latency, errors, token usage, search latency, tool failures, agent iterations and task completion. Application Insights would provide application telemetry, while Azure Monitor would provide broader metrics, logs, alerts and dashboards."

### Q5. How would you troubleshoot slow AI responses?

> "I would use distributed tracing to break the request into components and identify whether the latency is coming from APIM, application processing, Azure AI Search, tool execution, Azure OpenAI, or another dependency."

### Q6. What would you monitor specifically for RAG?

> "I would monitor retrieval latency, number of retrieved documents, Top-K, search errors, reranking latency, LLM latency and retrieval-quality metrics such as Precision@K and Recall@K separately from operational telemetry."

---

# 28. Senior-Level Scenario

### Interviewer:

> "Your Agentic AI application suddenly becomes slow in production. How will you investigate?"

A strong answer:

> "First, I would use Azure Monitor and Application Insights to check whether the degradation is application-wide or isolated to a dependency. I would inspect request latency, failure rates and distributed traces. Then I would break down the agent execution into LLM calls, Azure AI Search retrieval, tool calls and application processing. I would check Azure OpenAI throttling or latency, AI Search latency, Function/API dependencies and agent iteration counts. I would correlate everything using a request or correlation ID. Once the bottleneck is identified, I would apply the appropriate remediation, such as optimizing retrieval, reducing unnecessary LLM calls, handling throttling with controlled retries, or scaling the affected service."

---

# 29. The Key Mental Model

Remember this:

```text
Azure Monitor
    │
    ├── Metrics
    ├── Logs
    ├── Alerts
    ├── Dashboards
    └── Observability
             │
             ▼
      Application Insights
             │
             ├── Requests
             ├── Dependencies
             ├── Exceptions
             ├── Traces
             └── Application Performance
```

And for your **Agentic AI production architecture**:

> **APIM controls API traffic → Entra ID controls identity → Managed Identity secures Azure-to-Azure access → Key Vault manages unavoidable secrets → Content Safety handles AI safety → Application Insights observes the application → Azure Monitor provides broader monitoring, metrics, logs and alerting.**